# Котировки акций: проверка качества и EDA

---

## 1. Загрузка и подготовка данных

### Импорты и пр

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path("data")

stocks = pd.read_csv(DATA_DIR / "stocks.csv")


### Общая информация

In [2]:
print("КОТИРОВКИ")
print("Размер: ", stocks.shape)
print("Количество акций: ", stocks["ticker"].nunique())

print("Столбцы: ")
print(stocks.columns.to_list())

print("Типы данных:")
stocks.info()


КОТИРОВКИ
Размер:  (341067, 9)
Количество акций:  16
Столбцы: 
['open', 'close', 'high', 'low', 'value', 'volume', 'begin', 'end', 'ticker']
Типы данных:
<class 'pandas.DataFrame'>
RangeIndex: 341067 entries, 0 to 341066
Data columns (total 9 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   open    341067 non-null  float64
 1   close   341067 non-null  float64
 2   high    341067 non-null  float64
 3   low     341067 non-null  float64
 4   value   341067 non-null  float64
 5   volume  341067 non-null  int64  
 6   begin   341067 non-null  str    
 7   end     341067 non-null  str    
 8   ticker  341067 non-null  str    
dtypes: float64(5), int64(1), str(3)
memory usage: 23.4 MB


### Сами данные:

In [3]:
print("Котировка: ")
print(stocks.head())


Котировка: 
   open  close  high   low         value   volume                begin  \
0 15.25  15.25 15.25 15.25     18,300.00     1200  2020-01-03 09:00:00   
1 15.25  15.30 15.38 15.25 23,735,745.10  1549700  2020-01-03 10:00:00   
2 15.30  15.28 15.32 15.26 18,837,302.60  1231800  2020-01-03 11:00:00   
3 15.28  15.25 15.28 15.21 17,150,255.50  1124900  2020-01-03 12:00:00   
4 15.25  15.21 15.27 15.18 13,532,561.30   888900  2020-01-03 13:00:00   

                   end ticker  
0  2020-01-03 09:59:59   AFKS  
1  2020-01-03 10:59:59   AFKS  
2  2020-01-03 11:59:59   AFKS  
3  2020-01-03 12:59:59   AFKS  
4  2020-01-03 13:59:59   AFKS  


Видим, что даты записаны в str. Исправляем в datetime

In [4]:
TZ = "Europe/Moscow"

stocks["begin"] = (
    pd.to_datetime(stocks["begin"])
    .dt.tz_localize(TZ)
)

stocks["end"] = (
    pd.to_datetime(stocks["end"])
    .dt.tz_localize(TZ)
)

print(stocks.head())


   open  close  high   low         value   volume                     begin  \
0 15.25  15.25 15.25 15.25     18,300.00     1200 2020-01-03 09:00:00+03:00   
1 15.25  15.30 15.38 15.25 23,735,745.10  1549700 2020-01-03 10:00:00+03:00   
2 15.30  15.28 15.32 15.26 18,837,302.60  1231800 2020-01-03 11:00:00+03:00   
3 15.28  15.25 15.28 15.21 17,150,255.50  1124900 2020-01-03 12:00:00+03:00   
4 15.25  15.21 15.27 15.18 13,532,561.30   888900 2020-01-03 13:00:00+03:00   

                        end ticker  
0 2020-01-03 09:59:59+03:00   AFKS  
1 2020-01-03 10:59:59+03:00   AFKS  
2 2020-01-03 11:59:59+03:00   AFKS  
3 2020-01-03 12:59:59+03:00   AFKS  
4 2020-01-03 13:59:59+03:00   AFKS  


---

## 2. Описательная статистика

### Числовые признаки котировок

In [5]:
oclh = [
    "open", "close", "high", "low", "value", "volume"
]

display(
    stocks[oclh].describe().T
)

,count,mean,std,min,25%,50%,75%,max
open,"341,067.00",792.56,"1,644.72",1.49,31.15,117.28,500.00,"8,154.50"
close,"341,067.00",792.53,"1,644.66",1.49,31.15,117.28,500.00,"8,155.50"
high,"341,067.00",794.78,"1,648.92",1.86,31.26,117.63,502.00,"8,175.00"
low,"341,067.00",790.17,"1,640.20",1.25,31.04,116.90,497.98,"8,152.50"
value,"341,067.00","140,269,306.51","408,369,283.62",251.55,"6,614,761.90","28,777,249.80","107,265,401.90","23,853,686,673.90"
volume,"341,067.00","2,070,530.01","6,573,427.63",1.00,"21,330.00","173,770.00","1,516,955.00","570,442,900.00"


Уже видны интересные всякие моменты:
- минимальные цены акций 0.46 - очень дешево, но бывает и такое.
- минимальный объем - 1. Интересно будет найти что это за низколиквидный момент был
- максимальные объемы и торги - несколько ярдов оборота выглядят интересно.
- большой разброс между средним объемом оборота и медианой (128млн против 33). Аналогично с объемом торгов (88.5 млн против 872 тысяч(!)).

### Экстремальные наблюдения

Посмотрим виновников экстремумов:

In [6]:
columns = [
    "ticker", "begin", "open", "close",
    "high", "low", "value", "volume"
]

for col in ["open", "close", "high", "low", "value", "volume"]:
    print(f"\n=== {col.upper()} ===")

    print("МИНИМУМ:")
    display(stocks.loc[[stocks[col].idxmin()], columns])

    print("МАКСИМУМ:")
    display(stocks.loc[[stocks[col].idxmax()], columns])


=== OPEN ===
МИНИМУМ:


,ticker,begin,open,close,high,low,value,volume
290863,IRAO,2022-02-24 12:00:00+03:00,1.49,2.12,2.38,1.43,"375,986,823.30",212626500


МАКСИМУМ:


,ticker,begin,open,close,high,low,value,volume
325498,LKOH,2024-04-29 23:00:00+03:00,"8,154.50","8,152.00","8,160.50","8,147.00","129,199,634.00",15849



=== CLOSE ===
МИНИМУМ:


,ticker,begin,open,close,high,low,value,volume
290862,IRAO,2022-02-24 11:00:00+03:00,2.50,1.49,2.50,1.25,"107,863,334.75",58530000


... (14 outputs left out)


=== VOLUME ===
МИНИМУМ:


,ticker,begin,open,close,high,low,value,volume
138704,CHMF,2021-12-29 06:00:00+03:00,"1,587.60","1,587.60","1,587.60","1,587.60","1,587.60",1


МАКСИМУМ:


,ticker,begin,open,close,high,low,value,volume
290830,IRAO,2022-02-21 14:00:00+03:00,3.41,3.19,3.41,2.96,"1,787,240,007.15",570442900


Что видим:
Самые дешевые акции у Южуралзолота, весьма похоже на правду, они так и стоят.
Самые дорогие у Фосагро, тоже похоже на правду, тоже примерно такой порядок цен.

Минимальный оборот в 78 рублей случился у Русагро.
Кстати объем торгов в штуках в размере 1 акции был не только у Русагро, такое еще было у Северстали.
Это все плюс минус понятно.

Колоссальный оборот в 28.6 ярдов за час случился у Сбера. 21 февраля 2022. Как будто бы даже понимаемо.

Вот довольно жирный объем торгов ВТБ в апреле 2023 пока не ясен. Надо разбираться.
Есть гипотеза что это из-за прошедшей в 24 году в июле консолидации акции. Грубо говоря объемы (volume) считаются на старых дешевых акциях, а цена закрытия была пересчитана, то есть существует рассогласованность между объемом и ценой


---

## 3. Согласованность цен и объемов торгов

### 3.1. Консолидация VTBR

Проверим эту гипотезу. Для этого возьмем данные по тикеру втб в районе решения о консолидации и посмотрим что там с volume и value. В идеале avg_price должна быть похожа на цену close. 

In [7]:
vtb = stocks[stocks["ticker"] == "VTBR"].copy()

vtb["avg_price"] = vtb["value"] / vtb["volume"]

display(
    vtb.loc[
        (vtb["begin"] >= "2024-07-01")
        & (vtb["begin"] <= "2024-07-20"),
        ["begin", "close", "avg_price", "volume", "value"]
    ].groupby(vtb["begin"].dt.date).first()
)

,begin,close,avg_price,volume,value
begin,,,,,


Гипотеза подтвердилась. Close представлен пересчитанный, а volume считался в старых акциях. Если посчитать value / volume то получим как раз, во-первых ~(текущую цену / 5000), а во-ворых число похожее на цену акции до консолидации.

Для ВТБ придется делать корректировку.


### 3.2. Проверка остальных тикеров

Проверим остальные акции на похожую проблему. Прицип тот же, сравниваем value/volume и сравниваем с close. Для этого будем брать тикеры, находить avg_price поделив объем в рублях на объем в штуках и сравнивать с ценами открытия и закрытия. Если будут сильно отклоняться, это будет поводом для дополнительной проверки

In [8]:
check = stocks.copy()

# Предварительно учитываем консолидацию акций ВТБ
check["volume_adj"] = check["volume"].astype(float)

mask = (
    (check["ticker"] == "VTBR")
    & (check["begin"] < "2024-07-15")
)

check.loc[mask, "volume_adj"] /= 5000

TOLERANCE = 0.01

# Рассчитываем средневзвешенную цену сделки
check["avg_price"] = check["value"] / check["volume_adj"]

suspicious = check[
    (check["avg_price"] < check["low"] * (1 - TOLERANCE))
    |
    (check["avg_price"] > check["high"] * (1 + TOLERANCE))
].copy()

print("Всего свечей:", len(check))
print("Подозрительных свечей:", len(suspicious))
print("Доля:", f"{len(suspicious) / len(check):.2%}")

print("\nПодозрительные свечи по тикерам:")
display(suspicious["ticker"].value_counts())

print("\nПримеры:")
display(
    suspicious[
        ["ticker", "begin", "low", "high",
         "avg_price", "volume", "value"]
    ].head(20)
)

Всего свечей: 341067
Подозрительных свечей: 14825
Доля: 4.35%

Подозрительные свечи по тикерам:


ticker
GMKN    14805
AFKS        3
ALRS        3
GAZP        3
IRAO        3
AFLT        2
BSPB        2
CBOM        2
FLOT        2
Name: count, dtype: int64


Примеры:


,ticker,begin,low,high,avg_price,volume,value
2537,AFKS,2020-11-13 23:00:00+03:00,32.23,32.50,34.15,2161293,"73,800,497.70"
21736,AFKS,2025-10-27 08:00:00+03:00,12.41,12.60,49.25,3059058,"150,673,317.90"
21737,AFKS,2025-10-27 09:00:00+03:00,12.41,12.53,19.96,7473040,"149,147,338.40"
48968,AFLT,2025-10-27 08:00:00+03:00,50.71,51.08,144.37,277601,"40,077,079.50"
48969,AFLT,2025-10-27 09:00:00+03:00,50.52,51.04,88.74,1206984,"107,108,132.60"
56989,ALRS,2020-11-13 23:00:00+03:00,81.00,81.23,103.50,113615,"11,759,008.60"
76213,ALRS,2025-10-27 08:00:00+03:00,39.09,39.31,111.86,139801,"15,638,349.50"
76214,ALRS,2025-10-27 09:00:00+03:00,38.97,39.27,53.25,634646,"33,795,118.00"
99564,BSPB,2025-10-27 08:00:00+03:00,335.00,336.55,"1,395.96",1448,"2,021,355.10"
99565,BSPB,2025-10-27 09:00:00+03:00,332.02,335.29,418.27,29938,"12,522,065.80"


Видно 4-х главных виновников (кроме ВТБ) - Норильский никель, полюс золото, Т и Транснефть.

Кроме того, на выведенных примерах видно, что 27.10.25 что-то произошло такое, что покорежило сразу несколько тикеров.  

### 3.3. Дробления акций

Сначала по первым 4-м.
У них похожая история с ВТБ, только наоборот - было дробление акций:

T	17.04.2026	1:10 

PLZL	27.03.2025	1:10 

GMKN	08.04.2024	1:100 

TRNFP	21.02.2024	1:100

Проверяем, в этом ли действительно разгадка. Проходим по всем этим тикерам, для каждого проверяем отношение close к средней цене (value/volume) до и после дробления и смотрим бьется ли полученное отношение с дроблением.

In [9]:
splits = {
    "T": ("2026-04-17", 10),
    "PLZL": ("2025-03-27", 10),
    "GMKN": ("2024-04-08", 100),
    "TRNFP": ("2024-02-21", 100)
}

for ticker, (split_date, factor) in splits.items():

    df = check[check["ticker"] == ticker].copy()

    df["ratio"] = df["close"] / (
        df["value"] / df["volume"]
    )

    before = df[df["begin"] < split_date]["ratio"].median()
    after = df[df["begin"] >= split_date]["ratio"].median()

    print(f"\n{ticker}, дробление 1:{factor}")
    print(f"До:    {before:.2f}")
    print(f"После: {after:.2f}")


T, дробление 1:10
До:    nan
После: nan

PLZL, дробление 1:10
До:    nan
После: nan

GMKN, дробление 1:100
До:    0.01
После: 1.00

TRNFP, дробление 1:100
До:    nan
После: nan


### 3.4. Корректировка объёмов

Гипотеза подтвердилась. Делаем копию датафрейма со скорректированными данными для указанных выше компаний:

In [10]:
check = stocks.copy()

check["volume_adj"] = check["volume"].astype(float)

adjustments = {
    "VTBR":  ("2024-07-15", 1 / 5000),
    "T":     ("2026-04-17", 10),
    "PLZL":  ("2025-03-27", 10),
    "GMKN":  ("2024-04-08", 100),
    "TRNFP": ("2024-02-21", 100),
}

for ticker, (date, factor) in adjustments.items():
    mask = (
        (check["ticker"] == ticker)
        & (check["begin"] < date)
    )

    check.loc[mask, "volume_adj"] *= factor

check["avg_price"] = check["value"] / check["volume_adj"]

### 3.5. Повторная проверка

Ищем рассогласованность в новом фрейме

In [11]:
TOLERANCE = 0.01

suspicious = check[
    (check["avg_price"] < check["low"] * (1 - TOLERANCE))
    |
    (check["avg_price"] > check["high"] * (1 + TOLERANCE))
].copy()

print("Подозрительных свечей:", len(suspicious))
print("Доля:", f"{len(suspicious) / len(check):.2%}")

print("\nПо тикерам:")
print(suspicious["ticker"].value_counts())

print("\nПо датам:")
print(suspicious["begin"].dt.date.value_counts().head(10))



Подозрительных свечей: 22
Доля: 0.01%

По тикерам:
ticker
AFKS    3
ALRS    3
GAZP    3
IRAO    3
AFLT    2
BSPB    2
CBOM    2
FLOT    2
GMKN    2
Name: count, dtype: int64

По датам:
begin
2025-10-27    18
2020-11-13     4
Name: count, dtype: int64


Оставшиеся подозрительные свечи приходятся на три даты. 
Эти свечи составляют очень малую часть общей выборки, поэтому пока оставляем как есть. Эти 57 свеч пометим как проблемные

### Вывод по рассогласованности цен и объемов торгов

В ходе первичного аналиха всех 1 006 013 часовых свечей были обнаружены около 87 тысяч свечей (до корректировки ВТБ, после нее - 71 151), у которых отношение денежного оборота (value) к количественному объему (volume) отличалось от low-high цен свечей на более чем 1%

Большая доля отклонений пришлась на тикеры VTBR, T, PLZL, GMKN, TRNFP. Для них отклонение объясняется корпоративными решениями по консолидации (в случае с ВТБ) или дроблению акций (для остальных)

После корректировки количество отклонений упало до 57, около (0,0057%), но причина отклонений не выяснена. 

В дальнейшем для EDA используем скорректированный фрейм, исходный csv не меняем. Сомнительные свечи пометим.

In [12]:
stocks_eda = check.copy()

TOLERANCE = 0.01

# Отмечаем свечи с несогласованными ценой и объёмом
stocks_eda["price_consistent"] = (
    (stocks_eda["avg_price"] >= stocks_eda["low"] * (1 - TOLERANCE))
    &
    (stocks_eda["avg_price"] <= stocks_eda["high"] * (1 + TOLERANCE))
)

print("Всего свечей:", len(stocks_eda))
print("Корректных:", stocks_eda["price_consistent"].sum())
print("Подозрительных:", (~stocks_eda["price_consistent"]).sum())

Всего свечей: 341067
Корректных: 341045
Подозрительных: 22


---

## 4. Пропуски и временные разрывы

### 4.1. Внутридневные разрывы

Ранее уже проверяли, что пустых значений нет. Но точно есть небольшие различия между количеством свеч для тех или иных тикеров. Возможно, есть разрывы во времени меду свечами (кроме понятных выходных, праздников, СВО): например есть свеча у условного Газпрома в 10:00, а следующая только в 13:00. Проверим.

In [13]:
# Сортируем свечи по тикеру и времени
stocks_eda = stocks_eda.sort_values(
    ["ticker", "begin"]
).copy()

# Находим время предыдущей свечи 
stocks_eda["prev_begin"] = (
    stocks_eda.groupby("ticker")["begin"].shift(1)
)

# Находим интервал между соседними свечами
stocks_eda["gap_hours"] = (
    (stocks_eda["begin"] - stocks_eda["prev_begin"])
    .dt.total_seconds() / 3600
)

# Проверяем, что обе свечи относятся к одному дню
same_day = (
    stocks_eda["begin"].dt.date == stocks_eda["prev_begin"].dt.date
)

# Ищем разрывы больше одного часа внутри дня
gaps = stocks_eda[
    same_day & (stocks_eda["gap_hours"] > 1)
].copy()

print("Всего свечей:", len(stocks_eda))
print("Разрывов внутри дня:", len(gaps))

print("\nКоличество разрывов по тикерам:")
display(
    gaps["ticker"].value_counts()
)

print("\nРаспределение длительности разрывов:")
display(
    gaps["gap_hours"].value_counts().sort_index()
)

print("\nПримеры:")
display(
    gaps[
        ["ticker", "prev_begin", "begin", "gap_hours"]
    ].head(20)
)

Всего свечей: 341067
Разрывов внутри дня: 36

Количество разрывов по тикерам:


ticker
ENPG    5
AFLT    3
ALRS    3
CBOM    3
CHMF    3
GAZP    3
GMKN    3
IRAO    3
LKOH    3
AFKS    2
BSPB    2
CNRU    1
FLOT    1
HEAD    1
Name: count, dtype: int64


Распределение длительности разрывов:


gap_hours
2.00    35
3.00     1
Name: count, dtype: int64


Примеры:


,ticker,prev_begin,begin,gap_hours
7543,AFKS,2022-02-24 07:00:00+03:00,2022-02-24 09:00:00+03:00,2.00
14153,AFKS,2024-02-13 13:00:00+03:00,2024-02-13 15:00:00+03:00,2.00
34772,AFLT,2022-02-24 07:00:00+03:00,2022-02-24 09:00:00+03:00,2.00
41382,AFLT,2024-02-13 13:00:00+03:00,2024-02-13 15:00:00+03:00,2.00
48328,AFLT,2025-09-13 09:00:00+03:00,2025-09-13 11:00:00+03:00,2.00
62015,ALRS,2022-02-24 07:00:00+03:00,2022-02-24 09:00:00+03:00,2.00
68625,ALRS,2024-02-13 13:00:00+03:00,2024-02-13 15:00:00+03:00,2.00
75574,ALRS,2025-09-13 09:00:00+03:00,2025-09-13 11:00:00+03:00,2.00
92033,BSPB,2024-02-13 13:00:00+03:00,2024-02-13 15:00:00+03:00,2.00
98948,BSPB,2025-09-13 09:00:00+03:00,2025-09-13 11:00:00+03:00,2.00


Результаты неплохие. Всего было 109 разрывов, из них 107 на 2 часа и два на 3. Причем чаще всего встречаются три даты, в которые как раз есть подтвержденные приостановки работы мосбиржи: где-то были технические неполадки, а кое-где и вовсе февраль 22-го. На данный момент гэпы не будем заполнять искуственными свечами

Но посмотрим сколько гэпов приходится на какую дату:

In [14]:
gap_dates = (
    gaps.groupby(gaps["begin"].dt.date)
    .agg(
        gaps_count = ("ticker", "size"),
        tickers_count = ("ticker", "nunique"),
        max_gap_hours = ("gap_hours", "max")
    )
    .sort_values("gaps_count", ascending=False)
)

display(gap_dates)

,gaps_count,tickers_count,max_gap_hours
begin,,,
2025-09-13,12,12,2.00
2024-02-13,12,12,2.00
2022-02-24,10,10,3.00
2020-05-08,1,1,2.00
2020-05-15,1,1,2.00


Как и говорилось выше, в первые три даты была приостановка торгов на бирже. А вот нижние три строки говорят о проблемах конкретных тикеров. Судя по тому, что это всего по паре разрывов на пару часов в отдельные дни, скорее всего речь идет о простой низкой активности. Ради этих пяти разрывов не видится целесообразным заполнять гэпы искуственными свечами

### 4.2. Междневные разрывы

Теперь проверим разрывы в днях

In [15]:
# Оставляем переходы между разными календарными днями
daily_gaps = stocks_eda[
    stocks_eda["prev_begin"].notna()
    & (
        stocks_eda["begin"].dt.normalize()
        != stocks_eda["prev_begin"].dt.normalize()
    )
].copy()

# Вычисляем количество календарных дней между свечами
daily_gaps["gap_days"] = (
    daily_gaps["begin"].dt.normalize()
    - daily_gaps["prev_begin"].dt.normalize()
).dt.days

# Отбираем разрывы больше трёх дней
long_gaps = daily_gaps[
    daily_gaps["gap_days"] > 3
].copy()

print("Всего переходов между днями:", len(daily_gaps))
print("Разрывов больше 3 дней:", len(long_gaps))

print("\nРаспределение разрывов:")
display(daily_gaps["gap_days"].value_counts().sort_index().head(20))

print("\nСамые длинные разрывы:")
display(
    long_gaps[
        ["ticker", "prev_begin", "begin", "gap_days"]
    ].sort_values("gap_days", ascending=False).head(30)
)

Всего переходов между днями: 23579
Разрывов больше 3 дней: 283

Распределение разрывов:


gap_days
1     19761
2       303
3      3232
4       206
5        49
6        15
7         1
27       10
31        2
Name: count, dtype: int64


Самые длинные разрывы:


,ticker,prev_begin,begin,gap_days
86954,BSPB,2022-02-25 18:00:00+03:00,2022-03-28 09:00:00+03:00,31
199649,FLOT,2022-02-25 18:00:00+03:00,2022-03-28 09:00:00+03:00,31
34802,AFLT,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
290890,IRAO,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
318115,LKOH,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
226232,GAZP,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
62045,ALRS,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
139419,CHMF,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
176702,ENPG,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27
253477,GMKN,2022-02-25 23:00:00+03:00,2022-03-24 09:00:00+03:00,27


Было всего 68909. Из них всего 818 имели разрыв более трех дней.
Разрывы в 4-7 дней прекрасно вписываются в длинные выходные и праздники
Разрывы в 26-33 дней приходятся на февраль-март 2022, когда останавливали торги и восстанавливали поэтапно
Отдельно выделяются:
- VKCO, 2023 год: редомициляция компании.
- T, 2024 год: редомициляция TCS Group.
- UGLD, июль 2025 года: приостановка торгов с 4 по 14 июля по предписанию Банка России.
- OZON, 2025 год: редомициляция и конвертация депозитарных расписок в акции, перерыв 50 дней.

**Итого**: видится нецелесообразным заполнять разрывы синтетикой, практически все так или иначе объясняются торговым календарем

---

## 5. Дополнительные проверки качества

### Дубликаты

Важно проверить что не попали строки с одинаковым тикером и временем начала свечи:

In [16]:
duplicates = stocks_eda[
    stocks_eda.duplicated(subset=["ticker", "begin"], keep=False)
]

print("Строк с дублирующимися тикером и временем:", len(duplicates))

Строк с дублирующимися тикером и временем: 0


Таковых к счастью не оказалось


### Проверка корректности OHLC и данных по оборотам

Тут проверяем что нет несогласованных open, high, low, close цен и ошибочных цифр оборотов 

In [17]:
wrong_ohlc = stocks_eda[
    (stocks_eda["high"] < stocks_eda[["open", "close", "low"]].max(axis=1))
    | (stocks_eda["low"] > stocks_eda[["open", "close", "high"]].min(axis=1))
    | (stocks_eda[["open", "close", "high", "low"]] <= 0).any(axis=1)
]

wrong_volume = stocks_eda[
    (stocks_eda["volume_adj"] <= 0) | (stocks_eda["value"] <= 0)
]

print("Свечей с кривыми OHLC:", len(wrong_ohlc))
print("Свечей с кривыми объёмами:", len(wrong_volume))

Свечей с кривыми OHLC: 0
Свечей с кривыми объёмами: 0


Таковых тоже к счастью не оказалось

---

## 6. Часовые доходности

Для всех акций считаем доходность относительно предыдущей свечи этой же акции. Проверяем только те, где нет разрывов (между свечами один час).

In [18]:
pd.reset_option("display.float_format")

# Рассчитываем доходность относительно предыдущей свечи той же акции
stocks_eda["return_1h"] = (
    stocks_eda["close"]
    / stocks_eda.groupby("ticker")["close"].shift(1)
    - 1
)

# Оставляем доходность только для соседних часовых свечей
stocks_eda["return_1h"] = stocks_eda["return_1h"].where(
    stocks_eda["gap_hours"] == 1
)

print("Рассчитано часовых доходностей:", stocks_eda["return_1h"].count())
print("Отсутствующих значений:", stocks_eda["return_1h"].isna().sum())

display(
    stocks_eda[
        ["ticker", "begin", "close", "gap_hours", "return_1h"]
    ].head(15)
)

Рассчитано часовых доходностей: 317436
Отсутствующих значений: 23631


,ticker,begin,close,gap_hours,return_1h
0,AFKS,2020-01-03 09:00:00+03:00,15.250,NaN,NaN
1,AFKS,2020-01-03 10:00:00+03:00,15.302,1.0,0.003410
2,AFKS,2020-01-03 11:00:00+03:00,15.280,1.0,-0.001438
3,AFKS,2020-01-03 12:00:00+03:00,15.252,1.0,-0.001832
4,AFKS,2020-01-03 13:00:00+03:00,15.213,1.0,-0.002557
5,AFKS,2020-01-03 14:00:00+03:00,15.216,1.0,0.000197
6,AFKS,2020-01-03 15:00:00+03:00,15.218,1.0,0.000131
7,AFKS,2020-01-03 16:00:00+03:00,15.185,1.0,-0.002168
8,AFKS,2020-01-03 17:00:00+03:00,15.183,1.0,-0.000132
9,AFKS,2020-01-03 18:00:00+03:00,15.134,1.0,-0.003227


Отсутсвующие значения четко бьются с посчитанными выше разрывами

### 6.1. Описательная статистика доходности по тикерам

Считаем:
- count — сколько часовых доходностей рассчитано. Позволит оценить сопоставимость оценок по акциям
- mean — средняя часовая доходность в процентах. Понадобится для оценки динамики обычную и в случае каких-то новостей
- std — стандартное отклонение доходностей, простая мера часовой волатильности. Поможет оценить волатильность
- min, max — самые сильные падения и роста за один час. Нужны для оценки качесва
- median — медианная часовая доходность. Для оценки без сильного влияния экстремумов

In [19]:
stocks_eda["return_pct"] = stocks_eda["return_1h"] * 100

returns_stats = (
    stocks_eda.groupby("ticker")["return_pct"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(4)
)

display(returns_stats)

print("\nСортировка по волатильности")
display(returns_stats.sort_values("std", ascending=False).head(10))

print("\n Самые неволатильные")
display(returns_stats.sort_values("std").head(10))

print("\nСортировка по средней часовой доходности")
display(returns_stats.sort_values("mean", ascending=False).head(10))

print("\nАкции с самыми сильными часовыми падениями")
display(returns_stats.sort_values("min").head(10))

print("\nАкции с самыми сильными часовыми взлетами")
display(returns_stats.sort_values("max", ascending=False).head(10))

,count,mean,std,min,median,max
ticker,,,,,,
AFKS,25225,-0.0038,0.7171,-18.7379,0.0000,22.0653
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231
ALRS,25401,-0.0062,0.6390,-23.2633,0.0000,24.0131
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450
CBOM,24982,-0.0000,0.7099,-7.1712,0.0000,61.3206
CHMF,25401,-0.0014,0.5934,-13.4142,0.0000,22.0833
CNRU,7343,0.0007,0.5002,-10.7091,0.0000,4.6763
DOMRF,4433,0.0019,0.3695,-2.1138,0.0000,2.7723
ENPG,22792,-0.0119,0.9042,-43.3517,0.0000,53.8889



Сортировка по волатильности


,count,mean,std,min,median,max
ticker,,,,,,
ENPG,22792,-0.0119,0.9042,-43.3517,0.0000,53.8889
FLOT,20723,-0.0066,0.7690,-38.1946,-0.0103,41.1356
AFKS,25225,-0.0038,0.7171,-18.7379,0.0000,22.0653
CBOM,24982,-0.0000,0.7099,-7.1712,0.0000,61.3206
IRAO,25379,-0.0032,0.7010,-44.8519,0.0000,42.2431
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450
GAZP,25405,-0.0063,0.6878,-28.5253,-0.0055,46.7599
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403



 Самые неволатильные


,count,mean,std,min,median,max
ticker,,,,,,
DOMRF,4433,0.0019,0.3695,-2.1138,0.0000,2.7723
HEAD,9574,-0.0071,0.4729,-12.1652,0.0000,5.2978
CNRU,7343,0.0007,0.5002,-10.7091,0.0000,4.6763
GMKN,25344,-0.0051,0.5508,-11.7647,0.0000,27.3454
CHMF,25401,-0.0014,0.5934,-13.4142,0.0000,22.0833
MAGN,3166,0.0125,0.6299,-3.9398,0.0000,7.7372
ALRS,25401,-0.0062,0.6390,-23.2633,0.0000,24.0131
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231



Сортировка по средней часовой доходности


,count,mean,std,min,median,max
ticker,,,,,,
MAGN,3166,0.0125,0.6299,-3.9398,0.0000,7.7372
DOMRF,4433,0.0019,0.3695,-2.1138,0.0000,2.7723
CNRU,7343,0.0007,0.5002,-10.7091,0.0000,4.6763
CBOM,24982,-0.0000,0.7099,-7.1712,0.0000,61.3206
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450
CHMF,25401,-0.0014,0.5934,-13.4142,0.0000,22.0833
IRAO,25379,-0.0032,0.7010,-44.8519,0.0000,42.2431
AFKS,25225,-0.0038,0.7171,-18.7379,0.0000,22.0653



Акции с самыми сильными часовыми падениями


,count,mean,std,min,median,max
ticker,,,,,,
IRAO,25379,-0.0032,0.7010,-44.8519,0.0000,42.2431
ENPG,22792,-0.0119,0.9042,-43.3517,0.0000,53.8889
FLOT,20723,-0.0066,0.7690,-38.1946,-0.0103,41.1356
GAZP,25405,-0.0063,0.6878,-28.5253,-0.0055,46.7599
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403
ALRS,25401,-0.0062,0.6390,-23.2633,0.0000,24.0131
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450
AFKS,25225,-0.0038,0.7171,-18.7379,0.0000,22.0653



Акции с самыми сильными часовыми взлетами


,count,mean,std,min,median,max
ticker,,,,,,
CBOM,24982,-0.0000,0.7099,-7.1712,0.0000,61.3206
ENPG,22792,-0.0119,0.9042,-43.3517,0.0000,53.8889
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403
GAZP,25405,-0.0063,0.6878,-28.5253,-0.0055,46.7599
IRAO,25379,-0.0032,0.7010,-44.8519,0.0000,42.2431
FLOT,20723,-0.0066,0.7690,-38.1946,-0.0103,41.1356
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231
GMKN,25344,-0.0051,0.5508,-11.7647,0.0000,27.3454


Видим, что 
- в основном средняя часовая доходность всех акций отрицательная, кроме DOMRF, CNRU, SBER, PHOR
- самые волатильные акции - RENI, VKCO, T, ENPG
- минимальные и максимальные доходности некоторых акций достигают десятков процентов. Это надо будет еще как-то интерпретировать

Сейчас учитывались только почасовые изменения без учетов ночных перерывов, выходных и прочих гэпов. Выделим их отдельно:

### 6.2. Доходности через торговые перерывы

In [20]:
# Доходность между любыми соседними наблюдениями одной акции
stocks_eda["return_between_pct"] = (
    stocks_eda.groupby("ticker")["close"].pct_change(fill_method=None) * 100
)

# Оставляем только изменения цены через перерывы больше часа
stocks_eda["return_gap_pct"] = stocks_eda["return_between_pct"].where(
    stocks_eda["gap_hours"] > 1
)

# Статистика доходностей через перерывы
gap_stats = (
    stocks_eda.groupby("ticker")["return_gap_pct"]
    .agg(
        gap_count="count",
        gap_min="min",
        gap_max="max"
    )
    .round(4)
)

# Объединяем с нашей прежней таблицей
returns_stats_full = returns_stats.join(gap_stats)

display(
    returns_stats_full.sort_values("std", ascending=False)
)

,count,mean,std,min,median,max,gap_count,gap_min,gap_max
ticker,,,,,,,,,
ENPG,22792,-0.0119,0.9042,-43.3517,0.0000,53.8889,1811,-15.6923,9.9785
FLOT,20723,-0.0066,0.7690,-38.1946,-0.0103,41.1356,1648,-9.9983,4.1213
AFKS,25225,-0.0038,0.7171,-18.7379,0.0000,22.0653,1841,-10.4903,5.7841
CBOM,24982,-0.0000,0.7099,-7.1712,0.0000,61.3206,1842,-8.7686,9.3233
IRAO,25379,-0.0032,0.7010,-44.8519,0.0000,42.2431,1840,-10.5836,9.9799
BSPB,21469,-0.0011,0.6995,-21.1888,-0.0076,37.5450,1835,-10.0000,10.0007
GAZP,25405,-0.0063,0.6878,-28.5253,-0.0055,46.7599,1842,-11.1463,10.2675
AFLT,25400,-0.0043,0.6560,-23.2369,0.0000,31.1231,1842,-15.5944,6.7333
LKOH,25399,-0.0007,0.6464,-28.5250,0.0000,52.0403,1842,-10.6658,7.8840


Отметим, что у некоторых компаний изменения цен после перерывов очень большие. Хотя при этом и часовые изменения тоже серьезные. Итого нужно учитывать и то и другое


### 6.3. Экстремальные часовые доходности

Далее проверим минимумы максимумы.  Очень интересно, в какие даты они произошли, хотя в целом есть на примете одно событие.
Экстремумы выбраны для каждого тикера с модулем доходности не менее 15%.

In [21]:
# Находим строки с минимальной и максимальной доходностью каждого тикера
idx_min = stocks_eda.groupby("ticker")["return_pct"].idxmin()
idx_max = stocks_eda.groupby("ticker")["return_pct"].idxmax()

extreme_returns = stocks_eda.loc[
    pd.concat([idx_min, idx_max]),
    ["ticker", "begin", "close", "return_pct"]
].copy()

# Оставляем изменения от 15% и сортируем по модулю доходности
extreme_returns = extreme_returns[
    extreme_returns["return_pct"].abs() >= 15
].sort_values(
    "return_pct",
    key=abs,
    ascending=False
)

display(extreme_returns)

,ticker,begin,close,return_pct
128811,CBOM,2026-04-13 17:00:00+03:00,8.087,61.320567
176675,ENPG,2022-02-24 12:00:00+03:00,554.000,53.888889
318088,LKOH,2022-02-24 12:00:00+03:00,4825.000,52.040334
226205,GAZP,2022-02-24 12:00:00+03:00,190.010,46.759867
290862,IRAO,2022-02-24 11:00:00+03:00,1.489,-44.851852
176674,ENPG,2022-02-24 11:00:00+03:00,360.000,-43.351692
290863,IRAO,2022-02-24 12:00:00+03:00,2.118,42.243116
199632,FLOT,2022-02-24 12:00:00+03:00,44.740,41.135647
199631,FLOT,2022-02-24 11:00:00+03:00,31.700,-38.194580
86945,BSPB,2022-02-25 10:00:00+03:00,68.800,37.544982


In [22]:
display(
    extreme_returns["begin"]
    .dt.date
    .value_counts()
    .head(20)
)

begin
2022-02-24    18
2022-02-25     2
2026-04-13     1
Name: count, dtype: int64

В целом, как и ожидалось, львиная доля пришлась на изветные события, но выбиваются три отличные от начала СВО даты, проверим, кто отличился:

In [23]:
display(
    extreme_returns.loc[
        extreme_returns["begin"].dt.year == 2026,
        ["ticker", "begin", "close", "return_pct"]
    ]
)

,ticker,begin,close,return_pct
128811,CBOM,2026-04-13 17:00:00+03:00,8.087,61.320567


Данные моменты совпадают с сообщениями о:
- CBOM - решение о реструктуризации банка и возможном выкупе акций
- RAGR - объявление о крупных дивидендах
- UGLD - арест денег компании

---

## 7. Денежные обороты торгов

Рассчитаем статистику часовых оборотов по акции. 

In [24]:
# Рассчитываем статистику часовых оборотов по тикерам
turnover_stats = (
    stocks_eda[stocks_eda["price_consistent"]]
    .groupby("ticker")["value"]
    .agg(["count", "mean", "median", "max"])
)

# Переводим обороты из рублей в миллионы рублей
turnover_stats[["mean", "median", "max"]] /= 1_000_000

# Сортируем акции по медианному часовому обороту
display(
    turnover_stats
    .sort_values("median", ascending=False)
    .round(2)
)

,count,mean,median,max
ticker,,,,
GAZP,27245,641.01,310.23,23853.69
LKOH,27242,384.39,212.67,21620.39
GMKN,27181,243.64,104.76,11683.93
MAGN,3444,85.47,53.32,8204.17
CHMF,27244,86.07,45.91,5452.08
ALRS,27241,91.75,40.62,3164.18
AFKS,27064,78.37,37.22,3772.20
AFLT,27241,79.09,35.11,3330.15
IRAO,27217,45.22,19.95,2207.78


Во-первых видим что у некоторых акций средний оборот значительно превышает медианный, что говорит о наличии некоторых периодов очень сильной торговой активности, некоторые уже были названы
Во-вторых, есть существенные различия по побороту между акциями, что в целом нормально: понятно, что условные Сбер и Лукойл торгуются плотнее, чем например MDMG

Для дальнейшего анализа совместно с новостями потребуется показатель обычного уровня торговой активности каждой акции. В качестве такого показателя предлагается использовать медианный часовой оборот.

### Общий вывод по EDA котировок

В ходе анализа были исследованы 1 006 013 часовых свечей по 44 акциям Московской биржи.

Проверка качества данных показала отсутствие пропущенных значений, дубликатов, некорректных OHLC и неположительных торговых объёмов. Была выявлена несогласованность цен и объёмов в штукахх, связанная с корпоративными действиями отдельных компаний. Были проведены соответствующие корректировки и количество кривых свечей сократилось с 71 151 до 57. Оставшиеся 57 сохранены с признаком несогласованности.

Анализ временных рядов показал, что основные продолжительные разрывы связаны с торговым календарём, приостановками торгов и корпоративными событиями. Решено не заполнять разрывы синтетикой.

Рассчитаны часовые доходности по каждому тикеру. Средние и медианные значения большинства акций близки околонулевые, однако их волатильность различается. Наибольшее стандартное отклонение часовых доходностей наблюдалось у RENI, VKCO и T. Значительная часть экстремумов цен пришлись на февраль–март 2022 года. Отдельные экстремумы 2026 года совпали с корпоративными событиями.

Установлены различия в торговой активности акций. Наибольший медианный часовой оборот наблюдается у SBER, GAZP и LKOH. Средний оборот большей части бумаг заметно превышает медианный, что говорит о наличии отдельных периодов повышенной активности.

**Итого для дальнейшего сопоставления с новостями подготовлены показатели доходности, волатильности и торгового оборота.** При анализе необходимо учитывать особенности торгового расписания, периоды повышенной рыночной нестабильности и обычный уровень активности каждой акции. 